# Existing forest, forest restoration and mangrove carbon maps

This notebook brings together the carbon-benefit outputs for existing forest, restorable forest areas and mangrove patches.

The combined overview map keeps only forest restoration and mangroves because existing forest and restoration polygons can overlap in mixed land-use classes. The panel maps show each layer separately using a stacked A4-friendly layout.

In [ ]:
import os
import sys
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
matplotlib_config_dir = base_path / ".matplotlib"
matplotlib_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config_dir))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

pd.set_option("display.max_columns", 200)


## Paths


In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"
colour_scale_upper_percentile = 0.98

existing_forest_carbon_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit_existing_forests/existing_forest_patch_global_carbon_benefit_summary.gpkg"
forest_carbon_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit_restorable_forests/restorable_forest_patch_global_carbon_benefit_summary.gpkg"
mangrove_carbon_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit/fn_mangrove_patch_global_carbon_benefit_summary.gpkg"
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
administrative_boundaries_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg"

output_dir = base_path / "dphil_paper_3/results/co_benefits/carbon/forest_existing_mangrove_carbon_maps"
output_dir.mkdir(parents=True, exist_ok=True)

summary_table_path = output_dir / "existing_forest_restoration_mangrove_carbon_map_summary.csv"
combined_per_hectare_map_path = output_dir / "forest_restoration_mangrove_carbon_per_hectare_combined_map.png"
per_hectare_panel_map_path = output_dir / "existing_forest_restoration_mangrove_carbon_per_hectare_stacked_panel_map.png"
total_carbon_panel_map_path = output_dir / "existing_forest_restoration_mangrove_carbon_total_stacked_panel_map.png"

required_paths = [
    existing_forest_carbon_path,
    forest_carbon_path,
    mangrove_carbon_path,
    jamaica_boundary_path,
    administrative_boundaries_path,
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

print(f"Existing forest carbon input: {existing_forest_carbon_path}")
print(f"Forest restoration carbon input: {forest_carbon_path}")
print(f"Mangrove carbon input: {mangrove_carbon_path}")
print(f"Outputs: {output_dir}")

## Load and standardise carbon layers

The two source notebooks use slightly different column names. This cell standardises the fields used for mapping:

- `carbon_total_tonnes_c`: total patch-level carbon potential/benefit.
- `carbon_tonnes_c_per_ha`: carbon potential/benefit per hectare.
- `mapped_area_ha`: restorable area for forest restoration patches, and patch area for mangroves.


In [ ]:
existing_forest_carbon = gpd.read_file(existing_forest_carbon_path).to_crs(jamaica_metric_grid_crs)
forest_carbon = gpd.read_file(forest_carbon_path).to_crs(jamaica_metric_grid_crs)
mangrove_carbon = gpd.read_file(mangrove_carbon_path).to_crs(jamaica_metric_grid_crs)
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)
administrative_boundaries = gpd.read_file(administrative_boundaries_path, layer="admin1").to_crs(jamaica_metric_grid_crs)

existing_forest_carbon = existing_forest_carbon[
    existing_forest_carbon.geometry.notna() & ~existing_forest_carbon.geometry.is_empty
].copy()
forest_carbon = forest_carbon[forest_carbon.geometry.notna() & ~forest_carbon.geometry.is_empty].copy()
mangrove_carbon = mangrove_carbon[mangrove_carbon.geometry.notna() & ~mangrove_carbon.geometry.is_empty].copy()
jamaica_boundary = jamaica_boundary[jamaica_boundary.geometry.notna() & ~jamaica_boundary.geometry.is_empty].copy()
administrative_boundaries = administrative_boundaries[
    administrative_boundaries.geometry.notna() & ~administrative_boundaries.geometry.is_empty
].copy()

existing_forest_map_layer = existing_forest_carbon[
    [
        "Existing_forest_patch_id",
        "existing_forest_area_ha",
        "total_carbon_potential_with_nn_fill",
        "carbon_potential_per_existing_forest_ha_with_nn_fill",
        "carbon_potential_estimate_method",
        "geometry",
    ]
].rename(
    columns={
        "Existing_forest_patch_id": "source_patch_id",
        "existing_forest_area_ha": "mapped_area_ha",
        "total_carbon_potential_with_nn_fill": "carbon_total_tonnes_c",
        "carbon_potential_per_existing_forest_ha_with_nn_fill": "carbon_tonnes_c_per_ha",
        "carbon_potential_estimate_method": "carbon_estimate_method",
    }
)
existing_forest_map_layer["ecosystem"] = "Existing forest"

forest_map_layer = forest_carbon[
    [
        "Restorable_patch_id",
        "restorable_area_ha",
        "total_carbon_potential_with_nn_fill",
        "carbon_potential_per_restorable_ha_with_nn_fill",
        "carbon_potential_estimate_method",
        "geometry",
    ]
].rename(
    columns={
        "Restorable_patch_id": "source_patch_id",
        "restorable_area_ha": "mapped_area_ha",
        "total_carbon_potential_with_nn_fill": "carbon_total_tonnes_c",
        "carbon_potential_per_restorable_ha_with_nn_fill": "carbon_tonnes_c_per_ha",
        "carbon_potential_estimate_method": "carbon_estimate_method",
    }
)
forest_map_layer["ecosystem"] = "Forest restoration areas"

mangrove_map_layer = mangrove_carbon[
    [
        "Mangrove_ID",
        "patch_area_ha",
        "total_carbon_benefit_with_nn_fill",
        "carbon_benefit_per_patch_ha_with_nn_fill",
        "carbon_benefit_estimate_method",
        "geometry",
    ]
].rename(
    columns={
        "Mangrove_ID": "source_patch_id",
        "patch_area_ha": "mapped_area_ha",
        "total_carbon_benefit_with_nn_fill": "carbon_total_tonnes_c",
        "carbon_benefit_per_patch_ha_with_nn_fill": "carbon_tonnes_c_per_ha",
        "carbon_benefit_estimate_method": "carbon_estimate_method",
    }
)
mangrove_map_layer["ecosystem"] = "Mangrove patches"

map_layers = [existing_forest_map_layer, forest_map_layer, mangrove_map_layer]
for map_layer in map_layers:
    map_layer["carbon_total_tonnes_c"] = pd.to_numeric(
        map_layer["carbon_total_tonnes_c"], errors="coerce"
    )
    map_layer["carbon_tonnes_c_per_ha"] = pd.to_numeric(
        map_layer["carbon_tonnes_c_per_ha"], errors="coerce"
    )
    map_layer["mapped_area_ha"] = pd.to_numeric(map_layer["mapped_area_ha"], errors="coerce")

combined_per_hectare_values = pd.concat(
    [map_layer["carbon_tonnes_c_per_ha"] for map_layer in map_layers],
    ignore_index=True,
).replace([np.inf, -np.inf], np.nan).dropna()
shared_per_hectare_colour_max = float(
    np.ceil(combined_per_hectare_values.quantile(colour_scale_upper_percentile) / 100) * 100
)
layer_total_colour_max_values = [
    np.ceil(map_layer["carbon_total_tonnes_c"].quantile(colour_scale_upper_percentile) / 100_000) * 100_000
    for map_layer in map_layers
]
shared_total_colour_max = float(max(layer_total_colour_max_values))


def summarise_map_layer(map_layer, ecosystem_name):
    return {
        "ecosystem": ecosystem_name,
        "patch_count": len(map_layer),
        "mapped_area_ha": map_layer["mapped_area_ha"].sum(),
        "total_carbon_tonnes_c": map_layer["carbon_total_tonnes_c"].sum(),
        "area_weighted_mean_carbon_tonnes_c_per_ha": map_layer["carbon_total_tonnes_c"].sum()
        / map_layer["mapped_area_ha"].sum(),
        "median_patch_carbon_tonnes_c_per_ha": map_layer["carbon_tonnes_c_per_ha"].median(),
        "patches_with_nearest_neighbour_fill": int(
            (map_layer["carbon_estimate_method"] == "nearest_observed_patch_mean").sum()
        ),
    }


summary_table = pd.DataFrame(
    [
        summarise_map_layer(existing_forest_map_layer, "Existing forest"),
        summarise_map_layer(forest_map_layer, "Forest restoration areas"),
        summarise_map_layer(mangrove_map_layer, "Mangrove patches"),
    ]
)
summary_table.to_csv(summary_table_path, index=False)

display(summary_table)
print(f"Shared per-hectare colour scale max: {shared_per_hectare_colour_max:,.0f} tonnes C/ha")
print(f"Shared total-carbon colour scale max: {shared_total_colour_max:,.0f} tonnes C")

## Plot helpers


In [ ]:
carbon_colour_map = plt.get_cmap("YlGn")
per_hectare_colour_norm = Normalize(vmin=0, vmax=shared_per_hectare_colour_max)
total_colour_norm = Normalize(vmin=0, vmax=shared_total_colour_max)

administrative_boundary_handle = Line2D(
    [0],
    [0],
    color="#424242",
    linewidth=0.7,
    label="Administrative boundaries",
)
forest_handle = Patch(facecolor="#66BB6A", edgecolor="none", label="Forest restoration areas")
mangrove_handle = Patch(facecolor="#1B9E77", edgecolor="black", label="Mangrove patches")


def style_jamaica_map(axis, title_text, show_scale_and_north=True):
    axis.set_title(title_text, pad=6)
    axis.set_axis_off()
    axis.set_aspect("equal")
    if show_scale_and_north:
        Robyn_paper_2_defs.draw_scale_bar(
            axis,
            location=(0.88, 0.78),
            length_km=20,
            linewidth=0.6,
            label_offset=0.02,
            km_offset=0.01,
        )
        Robyn_paper_2_defs.draw_north_arrow(
            axis,
            location=(0.88, 0.86),
            size=0.05,
            fontsize=8,
            label_offset=0.02,
        )


def add_horizontal_colourbar(figure, axes, colour_norm, label_text, colourbar_pad=0.035, colourbar_axis=None):
    scalar_mappable = ScalarMappable(norm=colour_norm, cmap=carbon_colour_map)
    scalar_mappable.set_array([])
    if colourbar_axis is None:
        colourbar = figure.colorbar(
            scalar_mappable,
            ax=axes,
            orientation="horizontal",
            fraction=0.045,
            pad=colourbar_pad,
            shrink=0.7,
            extend="max",
        )
    else:
        colourbar = figure.colorbar(
            scalar_mappable,
            cax=colourbar_axis,
            orientation="horizontal",
            extend="max",
        )
    colourbar.set_label(label_text, fontsize=9)
    colourbar.ax.tick_params(labelsize=8)
    colourbar.ax.xaxis.get_offset_text().set_size(8)
    return colourbar


million_tonnes_carbon_formatter = FuncFormatter(lambda tick_value, tick_position: f"{tick_value / 1_000_000:g}")


def format_colourbar_as_million_tonnes_c(colourbar):
    colourbar.ax.xaxis.set_major_formatter(million_tonnes_carbon_formatter)
    colourbar.ax.xaxis.offsetText.set_visible(False)
    colourbar.ax.set_xlabel(colourbar.ax.get_xlabel().replace("tonnes C", "million tonnes C"), fontsize=9)
    return colourbar


def plot_map_base(axis):
    jamaica_boundary.boundary.plot(ax=axis, color="#757575", linewidth=0.45)
    administrative_boundaries.boundary.plot(ax=axis, color="#424242", linewidth=0.25)

## Option 1: combined map

This map uses a shared per-hectare colour scale. Forest restoration areas are shown as polygons. Mangroves are shown as their original patch polygons, with a thin black outline so small patches remain visible at national scale.


In [ ]:
figure, axis = plt.subplots(figsize=(10, 6))
forest_map_layer.plot(
    ax=axis,
    column="carbon_tonnes_c_per_ha",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_per_hectare_colour_max,
    linewidth=0,
    alpha=0.92,
)
mangrove_map_layer.plot(
    ax=axis,
    column="carbon_tonnes_c_per_ha",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_per_hectare_colour_max,
    linewidth=0.35,
    edgecolor="black",
    zorder=4,
)
plot_map_base(axis)
style_jamaica_map(axis, "Carbon potential per hectare: forest restoration and mangroves")
axis.legend(
    handles=[forest_handle, mangrove_handle, administrative_boundary_handle],
    loc="lower left",
    frameon=True,
)
add_horizontal_colourbar(
    figure,
    axis,
    per_hectare_colour_norm,
    "Carbon potential per hectare (tonnes C/ha)",
)
figure.savefig(combined_per_hectare_map_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved combined map: {combined_per_hectare_map_path}")

## Option 2: stacked per-hectare panel map

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(8.27, 11.69), constrained_layout=True)

existing_forest_map_layer.plot(
    ax=axes[0],
    column="carbon_tonnes_c_per_ha",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_per_hectare_colour_max,
    linewidth=0,
)
plot_map_base(axes[0])
style_jamaica_map(axes[0], "a) Existing forest")

forest_map_layer.plot(
    ax=axes[1],
    column="carbon_tonnes_c_per_ha",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_per_hectare_colour_max,
    linewidth=0,
)
plot_map_base(axes[1])
style_jamaica_map(axes[1], "b) Forest restoration areas")

mangrove_map_layer.plot(
    ax=axes[2],
    column="carbon_tonnes_c_per_ha",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_per_hectare_colour_max,
    linewidth=0.35,
    edgecolor="black",
    zorder=4,
)
plot_map_base(axes[2])
style_jamaica_map(axes[2], "c) Mangrove patches")

add_horizontal_colourbar(
    figure,
    axes,
    per_hectare_colour_norm,
    "Carbon potential per hectare (tonnes C/ha)",
)
figure.savefig(per_hectare_panel_map_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved stacked per-hectare panel map: {per_hectare_panel_map_path}")

## Option 3: stacked total-carbon panel map

This map shows total carbon per patch. It uses one shared colour scale across existing forest, forest restoration areas and mangrove patches, so the three panels can be compared directly. The shared scale is labelled in million tonnes C to avoid scientific-notation offsets.

In [ ]:
figure = plt.figure(figsize=(8.27, 11.69), constrained_layout=True)
figure_grid = figure.add_gridspec(4, 1, height_ratios=[1.0, 1.0, 1.0, 0.08])
existing_axis = figure.add_subplot(figure_grid[0, 0])
forest_axis = figure.add_subplot(figure_grid[1, 0])
mangrove_axis = figure.add_subplot(figure_grid[2, 0])
total_colourbar_axis = figure.add_subplot(figure_grid[3, 0])

existing_forest_map_layer.plot(
    ax=existing_axis,
    column="carbon_total_tonnes_c",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_total_colour_max,
    linewidth=0,
)
plot_map_base(existing_axis)
style_jamaica_map(existing_axis, "a) Existing forest total carbon")

forest_map_layer.plot(
    ax=forest_axis,
    column="carbon_total_tonnes_c",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_total_colour_max,
    linewidth=0,
)
plot_map_base(forest_axis)
style_jamaica_map(forest_axis, "b) Forest restoration total carbon")

mangrove_map_layer.plot(
    ax=mangrove_axis,
    column="carbon_total_tonnes_c",
    cmap=carbon_colour_map,
    vmin=0,
    vmax=shared_total_colour_max,
    linewidth=0.35,
    edgecolor="black",
    zorder=4,
)
plot_map_base(mangrove_axis)
style_jamaica_map(mangrove_axis, "c) Mangrove patch total carbon")

total_colourbar = add_horizontal_colourbar(
    figure,
    [existing_axis, forest_axis, mangrove_axis],
    total_colour_norm,
    "Patch total carbon (tonnes C)",
    colourbar_axis=total_colourbar_axis,
)
format_colourbar_as_million_tonnes_c(total_colourbar)

figure.savefig(total_carbon_panel_map_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved stacked total-carbon panel map: {total_carbon_panel_map_path}")

## Outputs


In [ ]:
output_files = pd.DataFrame(
    [
        {"description": "summary table", "path": summary_table_path},
        {"description": "combined per-hectare map for forest restoration and mangroves", "path": combined_per_hectare_map_path},
        {"description": "stacked per-hectare panel map", "path": per_hectare_panel_map_path},
        {"description": "stacked total-carbon panel map", "path": total_carbon_panel_map_path},
    ]
)
output_files["path"] = output_files["path"].astype(str)
display(output_files)